In [1]:
import requests

In [2]:
repo_owner = 'evidentlyai'
repo_name = 'docs'
relative_path_from_repo_name = 'archive/refs/heads'
repo_branch_name = 'main'

# We are actually saving the repo but having it in memory
zip_url = f'https://github.com/{repo_owner}/{repo_name}/{relative_path_from_repo_name}/{repo_branch_name}.zip'
# Download the repository ZIP and keep it in memory (RAM not ROM). so if kernel is lost download data is lost 
zip_response = requests.get(zip_url)

In [3]:
len(zip_response.content)

17544837

In [4]:
import io
import zipfile

zip_archived = zipfile.ZipFile(io.BytesIO(zip_response.content))
zip_archived

<zipfile.ZipFile file=<_io.BytesIO object at 0x71185013afc0> mode='r'>

In [5]:
filenames = zip_archived.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [6]:
file_to_open_path = 'docs-main/docs/platform/alerts.mdx'
md_file = zip_archived.open(file_to_open_path)
md_content = md_file.read().decode('utf8')
print(md_content)

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently Enterprise**.
</Check>

![](/images/alerts.png)

To enable alerts, open the Project and navigate to the "Alerts" in the left menu. You must set:

* A notification channel.

* An alert condition.

## Notification channels

You can choose between the following options:

* **Email**. Add email addresses to send alerts to.

* **Slack**. Add a Slack webhook.

* **Discord**. Add a Discord webhook.

## Alert conditions

### Failed tests

If you use Tests (conditional checks) in your Project, you can tie alerting to the failed Tests in a Test Suite. Toggle this option on the Alerts page. Evidently will set an alert to the defined channel if any of the Tests fail.

<Tip>
  **How to avoid alert fatigue?** Use the `is_critical` parameter to mark non-critical Test as Warnings. Setting it to `False` prevent alerts for those checks even if th

In [7]:
#!uv add python-frontmatter

In [8]:
import frontmatter

post = frontmatter.loads(md_content)
print(post.content[:100])

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **Evidently En


In [9]:
print(post.metadata)

{'title': 'Alerts', 'description': 'How to set up alerts.'}


In [10]:
file_to_open_path = 'docs-main/docs/platform/alerts.mdx'
print(file_to_open_path.split('/'))
print(file_to_open_path.split('/', maxsplit=2))
print(file_to_open_path.split('/', maxsplit=1))

['docs-main', 'docs', 'platform', 'alerts.mdx']
['docs-main', 'docs', 'platform/alerts.mdx']
['docs-main', 'docs/platform/alerts.mdx']


In [11]:
needed_file_to_open_path1 = file_to_open_path.split('/', maxsplit=1)[-1]  #get the last one from ['docs-main', 'docs/platform/alerts.mdx']  which is  docs/platform/alerts.mdx
print(needed_file_to_open_path1)

_, needed_file_to_open_path2 = file_to_open_path.split('/', maxsplit=1)  #['docs-main', 'docs/platform/alerts.mdx'] becomes _ and needed_file_to_open_path2 becomes 'docs/platform/alerts.mdx'
print(needed_file_to_open_path2)

docs/platform/alerts.mdx
docs/platform/alerts.mdx


In [12]:
#sowe need to do it for all the files in zip_archived.namelist() to look in the format below
#remember post = frontmatter.loads(md_content)
# structure of data we want for each file in zip_archived.namelist()

one_file_details = {
    'content': post.content[:50],
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': needed_file_to_open_path1
}

print(one_file_details)

{'content': '<Check>\n  Built-in alerting is a Pro feature avail', 'title': 'Alerts', 'description': 'How to set up alerts.', 'filename': 'docs/platform/alerts.mdx'}


In [13]:
def read_github_repository(repo_owner, repo_name, repo_branch_name='main'):    
    # #The code to iterate all the file list when I get the url of the github zip
    # repo_owner = 'evidentlyai'
    # repo_name = 'docs'
    # repo_branch_name = 'main'

    # We are actually saving the repo but having it in memory
    zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{repo_branch_name}.zip'
    # Download the repository ZIP and keep it in memory (RAM not ROM). If the kernel restarts, these in-memory Python objects are lost.
    zip_response = requests.get(zip_url)
    zip_response.raise_for_status()

    git_zip_files_data = []

    with zipfile.ZipFile(io.BytesIO(zip_response.content)) as contained_in_zip:

        for file_path in contained_in_zip.namelist():
            # Only ingest text/documentation files
            if file_path.endswith(('.md', '.mdx', '.txt')):

                with contained_in_zip.open(file_path) as file:
                    file_content= file.read().decode('utf-8', errors='replace')
                    
                    file_ref= frontmatter.loads(file_content)
                    g_zip_file_data= {
                        'content': file_ref.content,
                        'title': file_ref.metadata.get('title'),
                        'description': file_ref.metadata.get('description'),
                        'filename': file_path.split('/', 1)[-1]
                    }
                    git_zip_files_data.append(g_zip_file_data)
            

    return git_zip_files_data



In [14]:
repo_files = read_github_repository(
    repo_owner='evidentlyai',
    repo_name='docs',
    repo_branch_name='main'
)
len(repo_files)

95

In [15]:
#repo_files[2]

first_file = repo_files[6]

print("TITLE:")
print(first_file['title'])

print("\nFILENAME:")
print(first_file['filename'])

print("\nCONTENT:")
print(first_file['content'][:100])

TITLE:
Descriptors

FILENAME:
docs/library/descriptors.mdx

CONTENT:
To evaluate text data, like LLM inputs and outputs, you create **Descriptors**. This is a universal 


In [16]:
#!uv add gitsource
#uv add git+https://github.com/alexeygrigorev/gitsource.git  for another repo not saved in PyPI

In [17]:
from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx", "txt"},
)

files = reader.read()

print(f'Loaded {len(files)} files')

Loaded 95 files


In [18]:
files[6].filename

'docs/library/descriptors.mdx'

In [19]:
# iterate and parse all files. parse using gitsource functions still
documents = [f.parse() for f in files]      #[result_you_want for item in collection]  ie For every f in files, run f.parse(), and put the result into a new list called documents
#same as this

# documents = []
# for f in files:
#     parsed_document = f.parse()
#     documents.append(parsed_document)



len(documents)

95

In [21]:
documents[10]

{'title': 'Output formats',
 'description': 'How to export the evaluation results.',
 'content': 'You can view or export Reports in multiple formats.\n\n**Pre-requisites**:\n\n* You know how to [generate Reports](/docs/library/report).\n\n## Log to Workspace\n\nYou can save the computed Report in Evidently Cloud or your local workspace.\n\n```python\nws.add_run(project.id, my_eval, include_data=False)\n```\n\n<Info>\n  **Uploading evals**. Check Quickstart examples [for ML](/quickstart_ml) or [for LLM](/quickstart_llm) for a full workflow.\n</Info>\n\n## View in Jupyter notebook\n\nYou can directly render the visual summary of evaluation results in interactive Python environments like Jupyter notebook or Colab.\n\nAfter running the Report, simply call the resulting Python object:\n\n```python\nmy_report\n```\n\nThis will render the HTML object directly in the notebook cell.\n\n## HTML\n\nYou can also save this interactive visual Report as an HTML file to open in a browser:\n\n```python

In [ ]:
query = 'LLM as a Judge'

rag:
1. Search --- this now  (For this we are going to use his minsearch library. But we have many others like Elastic search or we can create ours)
2. Prompt
3. Send to llm

In [25]:
#using minsearch
#!uv add minsearch
from minsearch import Index

In [26]:
#keyword_fields - when we want to have an exact match
#text_fields - this is where we want to broadly search
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

# #same meaning as sql below
# SELECT document from index
# WHERE filename = 'exact_match'
# ORDER BY RELEVANCE(document, 'search query')
# LIMIT 10

index.fit(documents)

In [ ]:
index.search(query, num_results=3) #only 3 relevant documents should be found in search

[{'title': 'LLM as a judge',
  'description': 'How to create and evaluate an LLM judge.',
  'content': 'import CloudSignup from \'/snippets/cloud_signup.mdx\';\nimport CreateProject from \'/snippets/create_project.mdx\';\n\nIn this tutorial, we\'ll show how to evaluate text for custom criteria using LLM as the judge, and evaluate the LLM judge itself.\n\n<Info>\n  **This is a local example.** You will run and explore results using the open-source Python library. At the end, we’ll optionally show how to upload results to the Evidently Platform for easy exploration.\n</Info>\n\nWe\'ll explore two ways to use an LLM as a judge:\n\n- **Reference-based**. Compare new responses against a reference. This is useful for regression testing or whenever you have a "ground truth" (approved responses) to compare against.\n- **Open-ended**. Evaluate responses based on custom criteria, which helps evaluate new outputs when there\'s no reference available.\n\nWe will focus on demonstrating **how to cre